# Setup — Mount Drive & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
!pip install --quiet torch torchio SimpleITK

# Write the 3 updated files into /content

Same `unet3d_film.py`, `diffusion_training.py`, `ddim_sampling.py`, with three changes:
- **Bigger patch support** — no code change needed, the U-Net is fully convolutional, it's
  just a config choice below (`PATCH_SIZE`).
- **Classifier-free guidance (CFG) dropout** — during training, both anchors (T00 + T50) get
  randomly zeroed together some % of the time, so the model also learns the "no anchor" case.
  This is what makes `guidance_scale` do anything at generation time.
- **Multi-sample averaging** — `generate_averaged()` runs DDIM generation multiple times and
  averages the results, plus gives you a per-voxel uncertainty map.

This overwrites the older copies sitting in your `CSE499` Drive folder from a previous session
(those are the pre-upgrade versions) — running this cell always gets you the current version.

In [ ]:
%%writefile /content/unet3d_film.py
"""
3D U-Net with FiLM conditioning — noise predictor for the phase-conditioned
diffusion model.

What this network does at every training step:
  input  = concat([noisy_Txx, T00, T50])   -> 3 channels
  cond   = phase value (0.1-0.9) + diffusion timestep (0-1000)
  output = predicted noise, same shape as noisy_Txx (1 channel)

FiLM (Feature-wise Linear Modulation) is applied inside every residual
block: output = gamma * features + beta, where gamma/beta come from the
phase + timestep embedding. This is what makes the same network generate
T20 differently from T70 -- and know how noisy the current input is.

No attention module yet -- that's an optional add-on for later, only if
there's time left after this trains cleanly.
"""

import math
import torch
import torch.nn as nn


# ---------------------------------------------------------------------------
# Embeddings: turn a single number (phase, or timestep) into a vector
# ---------------------------------------------------------------------------

class SinusoidalEmbedding(nn.Module):
    """Standard sinusoidal position embedding, same idea as in Transformers
    and in DDPM's timestep embedding. Turns a scalar into a vector of sines
    and cosines at different frequencies, so the network can tell 0.31 apart
    from 0.29 easily."""

    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        # x: (B,) scalar values
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=x.device).float() / half
        )
        args = x[:, None].float() * freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, dim)


class ConditionEncoder(nn.Module):
    """Combines phase embedding + timestep embedding into one conditioning
    vector that every residual block will read from."""

    def __init__(self, emb_dim=128, cond_dim=256):
        super().__init__()
        self.phase_emb = SinusoidalEmbedding(emb_dim)
        self.time_emb = SinusoidalEmbedding(emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, cond_dim),
            nn.SiLU(),
            nn.Linear(cond_dim, cond_dim),
        )

    def forward(self, phase, timestep):
        # phase: (B,) float in [0,1]-ish, timestep: (B,) float/int
        pe = self.phase_emb(phase)
        te = self.time_emb(timestep)
        return self.mlp(torch.cat([pe, te], dim=-1))  # (B, cond_dim)


# ---------------------------------------------------------------------------
# FiLM-conditioned residual block
# ---------------------------------------------------------------------------

class FiLMResBlock3D(nn.Module):
    """Two 3D convs with a FiLM modulation in between, plus a skip connection.
    This is the basic repeated unit of the whole network."""

    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.act = nn.SiLU()

        # Produces gamma and beta for this block's channel width
        self.film = nn.Linear(cond_dim, out_ch * 2)

        # If channel count changes, skip connection needs a 1x1 conv to match
        self.skip = (
            nn.Conv3d(in_ch, out_ch, kernel_size=1)
            if in_ch != out_ch
            else nn.Identity()
        )

    def forward(self, x, cond):
        h = self.act(self.norm1(self.conv1(x)))

        gamma, beta = self.film(cond).chunk(2, dim=-1)  # each (B, out_ch)
        gamma = gamma[:, :, None, None, None]
        beta = beta[:, :, None, None, None]
        h = gamma * h + beta  # FiLM: output = gamma * feature + beta

        h = self.act(self.norm2(self.conv2(h)))
        return h + self.skip(x)


# ---------------------------------------------------------------------------
# Down/up sampling
# ---------------------------------------------------------------------------

class Downsample3D(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.Conv3d(ch, ch, kernel_size=4, stride=2, padding=1)

    def forward(self, x):
        return self.op(x)


class Upsample3D(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.ConvTranspose3d(ch, ch, kernel_size=4, stride=2, padding=1)

    def forward(self, x):
        return self.op(x)


# ---------------------------------------------------------------------------
# The full U-Net
# ---------------------------------------------------------------------------

class UNet3DFiLM(nn.Module):
    """
    in_channels=3  -> noisy_Txx (1) + T00 (1) + T50 (1), concatenated
    out_channels=1 -> predicted noise, same shape as noisy_Txx

    base_ch controls model size -- start small (16 or 24) given the small
    dataset and GPU memory limits from patch-based training.
    """

    def __init__(self, in_channels=3, out_channels=1, base_ch=16, cond_dim=256):
        super().__init__()
        self.cond_encoder = ConditionEncoder(cond_dim=cond_dim)

        ch1, ch2, ch3, ch4 = base_ch, base_ch * 2, base_ch * 4, base_ch * 8

        # Encoder
        self.in_conv = nn.Conv3d(in_channels, ch1, kernel_size=3, padding=1)
        self.enc1 = FiLMResBlock3D(ch1, ch1, cond_dim)
        self.down1 = Downsample3D(ch1)

        self.enc2 = FiLMResBlock3D(ch1, ch2, cond_dim)
        self.down2 = Downsample3D(ch2)

        self.enc3 = FiLMResBlock3D(ch2, ch3, cond_dim)
        self.down3 = Downsample3D(ch3)

        # Bottleneck (this is where Phase-Aware Attention would go later)
        self.bottleneck = FiLMResBlock3D(ch3, ch4, cond_dim)

        # Decoder (mirrors encoder, with skip connections)
        self.up3 = Upsample3D(ch4)
        self.dec3 = FiLMResBlock3D(ch4 + ch3, ch3, cond_dim)

        self.up2 = Upsample3D(ch3)
        self.dec2 = FiLMResBlock3D(ch3 + ch2, ch2, cond_dim)

        self.up1 = Upsample3D(ch2)
        self.dec1 = FiLMResBlock3D(ch2 + ch1, ch1, cond_dim)

        self.out_conv = nn.Conv3d(ch1, out_channels, kernel_size=1)

    def forward(self, x, phase, timestep):
        """
        x:        (B, 3, D, H, W)  -- noisy_Txx + T00 + T50 concatenated
        phase:    (B,)             -- e.g. 0.3 for T30
        timestep: (B,)             -- e.g. 437 for a 1000-step schedule
        """
        cond = self.cond_encoder(phase, timestep)

        x0 = self.in_conv(x)
        e1 = self.enc1(x0, cond)          # skip 1
        x1 = self.down1(e1)

        e2 = self.enc2(x1, cond)          # skip 2
        x2 = self.down2(e2)

        e3 = self.enc3(x2, cond)          # skip 3
        x3 = self.down3(e3)

        b = self.bottleneck(x3, cond)

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1), cond)

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1), cond)

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1), cond)

        return self.out_conv(d1)


# ---------------------------------------------------------------------------
# Quick sanity check -- confirms shapes work before you touch real data
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    model = UNet3DFiLM(in_channels=3, out_channels=1, base_ch=16)

    # A small patch, not a full volume -- full volumes won't fit in GPU memory.
    # 64^3 is a reasonable starting patch size for a P100.
    batch_size = 2
    patch = 64
    x = torch.randn(batch_size, 3, patch, patch, patch)
    phase = torch.tensor([0.3, 0.7])
    timestep = torch.tensor([437.0, 120.0])

    out = model(x, phase, timestep)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"Input shape:  {tuple(x.shape)}")
    print(f"Output shape: {tuple(out.shape)}")
    print(f"Parameters:   {n_params:,}")
    assert out.shape == (batch_size, 1, patch, patch, patch), "Shape mismatch!"
    print("Sanity check passed: output shape matches input spatial shape.")


In [ ]:
%%writefile /content/diffusion_training.py
"""
Diffusion training loop for the phase-conditioned 3D U-Net.

This is Section 9 of the architecture notes, turned into runnable code:

  1. Pick a real intermediate phase (e.g. T30) from a training patient.
  2. Pick a random noise level t (0-999).
  3. Add exactly that much Gaussian noise to T30.
  4. Feed [noisy_T30, T00, T50] into the U-Net, with phase=0.3, timestep=t.
  5. U-Net predicts the noise that was added.
  6. Loss = MSE(predicted noise, actual noise).
  7. Backprop, update weights. Repeat across all patients/phases/noise levels.

Expects data laid out as:
  <data_root>/case<N>/T00.npy, T10.npy, ..., T90.npy         (real scans)
  <data_root>/synthetic/case<N>/T10_synth.npy, ...           (optional, augmentation)

Run modes:
  --selftest   builds tiny dummy data and runs 2 mini-epochs, to confirm the
               whole loop works end-to-end before touching real data or GPUs.
  (default)    real training run against --data-root.
"""

import argparse
import os
import random
import shutil
import tempfile

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from unet3d_film import UNet3DFiLM

INTERMEDIATE_PHASES = ["T10", "T20", "T30", "T40", "T60", "T70", "T80", "T90"]


# ---------------------------------------------------------------------------
# Noise schedule (standard DDPM linear beta schedule)
# ---------------------------------------------------------------------------

class NoiseScheduler:
    """Precomputes the noise schedule and handles the forward (noising)
    process. The reverse (denoising/sampling) process is a separate script --
    this one only needs forward noising to create training examples."""

    def __init__(self, num_timesteps=1000, beta_start=1e-4, beta_end=2e-2, device="cpu"):
        self.T = num_timesteps
        betas = torch.linspace(beta_start, beta_end, num_timesteps, device=device)
        alphas = 1.0 - betas
        self.alpha_bars = torch.cumprod(alphas, dim=0)  # (T,)

    def add_noise(self, x0, t):
        """x0: (B, 1, D, H, W) clean patch. t: (B,) long timestep indices.
        Returns (noisy_x, noise) -- both same shape as x0."""
        alpha_bar_t = self.alpha_bars[t].view(-1, 1, 1, 1, 1)  # (B,1,1,1,1)
        noise = torch.randn_like(x0)
        noisy_x = torch.sqrt(alpha_bar_t) * x0 + torch.sqrt(1 - alpha_bar_t) * noise
        return noisy_x, noise


# ---------------------------------------------------------------------------
# Dataset: real (+ optional synthetic) intermediate phases, patch-cropped
# ---------------------------------------------------------------------------

class PhasePatchDataset(Dataset):
    """Each item = one (patient, phase) pair, patch-cropped to a fixed size.
    Every item also carries that patient's T00 and T50 anchors, cropped from
    the exact same spatial location."""

    def __init__(self, data_root, case_ids, patch_size=64,
                 synthetic_root=None, synthetic_case_ids=None):
        self.data_root = data_root
        self.patch_size = patch_size
        self.samples = []  # list of (case_id, phase_name, is_synthetic)
        self._volume_cache = {}  # path -> np.ndarray, avoids re-reading Drive every __getitem__

        for case_id in case_ids:
            for phase in INTERMEDIATE_PHASES:
                path = self._real_path(case_id, phase)
                if os.path.exists(path):
                    self.samples.append((case_id, phase, False))

        if synthetic_root is not None:
            self.synthetic_root = synthetic_root
            for case_id in (synthetic_case_ids or case_ids):
                for phase in INTERMEDIATE_PHASES:
                    path = self._synth_path(case_id, phase)
                    if os.path.exists(path):
                        self.samples.append((case_id, phase, True))

        if not self.samples:
            raise RuntimeError(
                f"No training samples found under {data_root}. "
                "Check that case folders and .npy files exist."
            )

    def _real_path(self, case_id, phase):
        return os.path.join(self.data_root, f"case{case_id}", f"{phase}.npy")

    def _synth_path(self, case_id, phase):
        return os.path.join(self.synthetic_root, f"case{case_id}", f"{phase}_synth.npy")

    def _anchor_path(self, case_id, name):
        return os.path.join(self.data_root, f"case{case_id}", f"{name}.npy")

    def _load_cached(self, path):
        """Loads a .npy volume from Drive once, keeps it in RAM for every
        later access. On Colab, Drive reads are network I/O with real
        latency -- re-reading the same file every __getitem__ call (which
        happens once per sample per epoch, and the same handful of case
        volumes get reused across many samples/epochs) is what starves the
        GPU. Total cache size across the whole dataset is a few GB at most
        for DIR-Lab's volume sizes, well within Colab's ~12-13GB system RAM."""
        if path not in self._volume_cache:
            self._volume_cache[path] = np.load(path).astype(np.float32)
        return self._volume_cache[path]

    @staticmethod
    def _phase_to_float(phase_name):
        # "T30" -> 0.3
        return int(phase_name[1:]) / 100.0

    def _random_crop_coords(self, shape):
        p = self.patch_size
        coords = []
        for dim in shape:
            if dim <= p:
                coords.append(0)  # volume smaller than patch: start at 0, pad later
            else:
                coords.append(random.randint(0, dim - p))
        return coords

    def _crop(self, vol, coords):
        p = self.patch_size
        d0, h0, w0 = coords
        patch = vol[d0:d0 + p, h0:h0 + p, w0:w0 + p]
        # pad with zeros if volume was smaller than patch in any dimension
        pad = [(0, max(0, p - patch.shape[i])) for i in range(3)]
        if any(a or b for a, b in pad):
            patch = np.pad(patch, pad, mode="constant", constant_values=0)
        return patch

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        case_id, phase, is_synthetic = self.samples[idx]

        target_path = self._synth_path(case_id, phase) if is_synthetic else self._real_path(case_id, phase)
        target = self._load_cached(target_path)
        t00 = self._load_cached(self._anchor_path(case_id, "T00"))
        t50 = self._load_cached(self._anchor_path(case_id, "T50"))

        coords = self._random_crop_coords(target.shape)
        target_patch = self._crop(target, coords)
        t00_patch = self._crop(t00, coords)
        t50_patch = self._crop(t50, coords)

        # Data is already normalized to [-1, 1] by the preprocessing pipeline --
        # pass through unchanged. (Do NOT clip/1000 here -- that was a past bug:
        # dividing already-normalized values by 1000 again crushes them toward
        # zero, destroying the signal. If you ever swap in raw/un-normalized
        # .npy files, THIS is where you'd reintroduce a real HU->[-1,1] mapping.)
        def norm(x):
            return x

        target_patch = norm(target_patch)
        t00_patch = norm(t00_patch)
        t50_patch = norm(t50_patch)

        return {
            "target": torch.from_numpy(target_patch).unsqueeze(0),  # (1,D,H,W)
            "t00": torch.from_numpy(t00_patch).unsqueeze(0),
            "t50": torch.from_numpy(t50_patch).unsqueeze(0),
            "phase": torch.tensor(self._phase_to_float(phase), dtype=torch.float32),
        }


# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------

def train(model, dataloader, scheduler, device, epochs, lr, checkpoint_dir=None,
          cond_dropout_prob=0.15, use_amp=False, start_epoch=0):
    """
    cond_dropout_prob: probability of zeroing BOTH t00 and t50 together for a
        given training sample (classifier-free guidance training). Set to 0
        to disable and reproduce the old unconditional-free behavior exactly.
        Dropping both anchors together (not independently) matters -- the
        model should learn a clean "fully guided" vs "fully unguided"
        contrast, not weird half-states it'll never see at inference time.
    use_amp: enables mixed-precision training (torch.cuda.amp). Frees up GPU
        memory, which is what actually makes a bigger patch_size affordable
        on a T4. No effect on CPU.
    start_epoch: for resuming training (e.g. after loading a checkpoint and
        continuing with a bigger patch_size / cond_dropout_prob than the
        checkpoint was originally trained with). Only affects logging and
        saved filenames, not the optimizer state -- see load_state_dict call
        at the CLI level if you need true optimizer-state resume.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.to(device)
    model.train()

    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    history = []
    for epoch in range(epochs):
        running_loss = 0.0
        n_batches = 0

        for batch in dataloader:
            target = batch["target"].to(device)
            t00 = batch["t00"].to(device)
            t50 = batch["t50"].to(device)
            phase = batch["phase"].to(device)

            B = target.shape[0]
            t = torch.randint(0, scheduler.T, (B,), device=device).long()

            noisy_target, noise = scheduler.add_noise(target, t)

            # Classifier-free guidance: per-sample, drop BOTH anchors together
            # so the model also learns the unconditional (no-anchor) case.
            if cond_dropout_prob > 0:
                drop_mask = (torch.rand(B, device=device) < cond_dropout_prob).view(B, 1, 1, 1, 1)
                t00_in = torch.where(drop_mask, torch.zeros_like(t00), t00)
                t50_in = torch.where(drop_mask, torch.zeros_like(t50), t50)
            else:
                t00_in, t50_in = t00, t50

            model_input = torch.cat([noisy_target, t00_in, t50_in], dim=1)  # (B,3,D,H,W)

            with torch.amp.autocast("cuda", enabled=use_amp):
                predicted_noise = model(model_input, phase, t.float())
                loss = nn.functional.mse_loss(predicted_noise, noise)

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            n_batches += 1

        avg_loss = running_loss / max(n_batches, 1)
        history.append(avg_loss)
        global_epoch = start_epoch + epoch + 1
        print(f"Epoch {global_epoch} (run epoch {epoch + 1}/{epochs}) -- avg MSE loss: {avg_loss:.5f}")

        if checkpoint_dir:
            os.makedirs(checkpoint_dir, exist_ok=True)
            torch.save(model.state_dict(), os.path.join(checkpoint_dir, f"epoch_{global_epoch}.pt"))

    return history


# ---------------------------------------------------------------------------
# Self-test: builds tiny dummy data, confirms the whole loop runs end-to-end
# ---------------------------------------------------------------------------

def run_selftest():
    print("Running self-test with dummy data (no real CT data needed)...\n")
    tmp_dir = tempfile.mkdtemp()
    try:
        case_ids = [1, 2]
        vol_shape = (48, 48, 48)  # small on purpose, just to test plumbing

        for case_id in case_ids:
            case_dir = os.path.join(tmp_dir, f"case{case_id}")
            os.makedirs(case_dir, exist_ok=True)
            for name in ["T00", "T50"] + INTERMEDIATE_PHASES:
                arr = np.random.uniform(-1000, 1000, size=vol_shape).astype(np.float32)
                np.save(os.path.join(case_dir, f"{name}.npy"), arr)

        dataset = PhasePatchDataset(data_root=tmp_dir, case_ids=case_ids, patch_size=32)
        dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

        model = UNet3DFiLM(in_channels=3, out_channels=1, base_ch=8)  # tiny, just for speed
        scheduler = NoiseScheduler(num_timesteps=1000)

        history = train(model, dataloader, scheduler, device="cpu", epochs=2, lr=1e-3)

        assert len(history) == 2, "Expected 2 epochs of loss history"
        assert all(h == h for h in history), "Loss became NaN"
        print("\nSelf-test passed: data loads, patches crop correctly, "
              "model trains, loss is a real number.")
    finally:
        shutil.rmtree(tmp_dir)


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--selftest", action="store_true",
                         help="Run with dummy data to sanity-check the pipeline")
    parser.add_argument("--data-root", type=str, default="dirlab_preprocessed")
    parser.add_argument("--synthetic-root", type=str, default=None)
    parser.add_argument("--case-ids", type=int, nargs="+", default=[1, 2, 5, 6, 7, 8, 10])
    parser.add_argument("--patch-size", type=int, default=64)
    parser.add_argument("--batch-size", type=int, default=2)
    parser.add_argument("--epochs", type=int, default=2)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--base-ch", type=int, default=16)
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--checkpoint-dir", type=str, default="checkpoints")
    parser.add_argument("--cond-dropout-prob", type=float, default=0.15,
                         help="Probability of zeroing both anchors together, for classifier-free "
                              "guidance training. Set to 0 to disable.")
    parser.add_argument("--amp", action="store_true",
                         help="Enable mixed-precision training to free up GPU memory "
                              "(needed to afford a bigger --patch-size on a T4).")
    parser.add_argument("--resume-from", type=str, default=None,
                         help="Path to an existing .pt checkpoint to continue training from "
                              "(e.g. epoch_50.pt), instead of starting from random init. "
                              "Weights load fine even if --patch-size differs from the original "
                              "run, since the U-Net is fully convolutional. NOTE: a checkpoint "
                              "trained with cond_dropout_prob=0 has never seen zeroed anchors -- "
                              "expect it to need several epochs to adapt once you turn dropout on.")
    parser.add_argument("--start-epoch", type=int, default=0,
                         help="Epoch number to resume logging/checkpoint naming from. "
                              "Set this to match the epoch count of --resume-from.")
    args = parser.parse_args()

    if args.selftest:
        run_selftest()
    else:
        dataset = PhasePatchDataset(
            data_root=args.data_root,
            case_ids=args.case_ids,
            patch_size=args.patch_size,
            synthetic_root=args.synthetic_root,
        )
        dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True)

        model = UNet3DFiLM(in_channels=3, out_channels=1, base_ch=args.base_ch)
        if args.resume_from:
            model.load_state_dict(torch.load(args.resume_from, map_location=args.device))
            print(f"Resumed weights from {args.resume_from}")
        scheduler = NoiseScheduler(num_timesteps=1000, device=args.device)

        print(f"Training samples: {len(dataset)}")
        train(model, dataloader, scheduler, device=args.device,
              epochs=args.epochs, lr=args.lr, checkpoint_dir=args.checkpoint_dir,
              cond_dropout_prob=args.cond_dropout_prob, use_amp=args.amp,
              start_epoch=args.start_epoch)


In [ ]:
%%writefile /content/ddim_sampling.py
"""
DDIM sampling -- turns a TRAINED model into an actual generated lung phase.

Training (diffusion_training.py) only teaches the model to guess noise.
This script is the other half: start from pure random noise, and repeatedly
ask the trained model "what noise is this?", subtract a bit, and repeat --
until a clean, generated phase comes out the other end.

This is Section 8 of the architecture notes ("The Noise Subtractor"),
using DDIM instead of full DDPM so it only takes ~50 steps instead of 1000.

Usage:
  --selftest    builds a tiny untrained model and dummy anchors, runs the
                sampling loop, and confirms the output shape/values make
                sense -- no real checkpoint or CT data needed.
  (default)     loads a real checkpoint from diffusion_training.py and
                generates a real phase from real T00/T50 anchors.
"""

import argparse
import os

import numpy as np
import torch

from unet3d_film import UNet3DFiLM
from diffusion_training import NoiseScheduler


# ---------------------------------------------------------------------------
# DDIM sampler
# ---------------------------------------------------------------------------

class DDIMSampler:
    """Wraps a trained model + noise schedule and runs the reverse
    (denoising) process using DDIM's larger, calculated steps instead of
    DDPM's 1000 tiny ones."""

    def __init__(self, model, scheduler: NoiseScheduler, num_steps=50, device="cpu"):
        self.model = model.to(device).eval()
        self.scheduler = scheduler
        self.device = device

        # Pick num_steps evenly spaced timesteps out of the full schedule,
        # e.g. 50 steps out of 1000 -> [980, 960, ..., 20, 0]
        full_T = scheduler.T
        step_indices = np.linspace(0, full_T - 1, num_steps).round().astype(int)
        self.timesteps = list(step_indices[::-1])  # descending: noisy -> clean

    @torch.no_grad()
    def generate(self, t00, t50, phase, volume_shape, guidance_scale=1.0):
        """
        t00, t50: (1, 1, D, H, W) tensors -- the clean boundary anchors
        phase:    float, e.g. 0.3 for T30
        volume_shape: (D, H, W) -- shape of the phase to generate
        guidance_scale: classifier-free guidance strength. 1.0 = no guidance
            (identical to the old behavior, single forward pass per step).
            >1.0 runs the model TWICE per step -- once with real anchors,
            once with anchors zeroed out -- and extrapolates away from the
            unconditional prediction. Typical useful range is ~1.5-3.0.
            IMPORTANT: this only does something meaningful if the model
            checkpoint was actually trained with cond_dropout_prob > 0 in
            diffusion_training.py. A checkpoint trained with dropout=0 has
            never seen zeroed anchors, so the "unconditional" branch is
            out-of-distribution garbage and guidance_scale != 1.0 will hurt,
            not help, until you retrain (or resume-train) with dropout on.

        Returns: (1, 1, D, H, W) tensor -- the generated phase
        """
        device = self.device
        x = torch.randn((1, 1, *volume_shape), device=device)  # start: pure noise
        phase_t = torch.tensor([phase], device=device, dtype=torch.float32)

        use_guidance = guidance_scale != 1.0
        if use_guidance:
            zero_t00 = torch.zeros_like(t00)
            zero_t50 = torch.zeros_like(t50)

        for i, t_cur in enumerate(self.timesteps):
            t_prev = self.timesteps[i + 1] if i + 1 < len(self.timesteps) else -1
            t_batch = torch.tensor([t_cur], device=device, dtype=torch.float32)

            model_input = torch.cat([x, t00, t50], dim=1)  # (1,3,D,H,W)
            predicted_noise = self.model(model_input, phase_t, t_batch)

            if use_guidance:
                uncond_input = torch.cat([x, zero_t00, zero_t50], dim=1)
                noise_uncond = self.model(uncond_input, phase_t, t_batch)
                predicted_noise = noise_uncond + guidance_scale * (predicted_noise - noise_uncond)

            alpha_bar_t = self.scheduler.alpha_bars[t_cur]
            alpha_bar_prev = self.scheduler.alpha_bars[t_prev] if t_prev >= 0 else torch.tensor(1.0, device=device)

            # Predict the clean image from the current noisy one + predicted noise
            x0_pred = (x - torch.sqrt(1 - alpha_bar_t) * predicted_noise) / torch.sqrt(alpha_bar_t)
            x0_pred = torch.clamp(x0_pred, -1.0, 1.0)  # keep values in a sane range

            if t_prev >= 0:
                # Deterministic DDIM step (eta=0): move to the less-noisy image
                x = torch.sqrt(alpha_bar_prev) * x0_pred + torch.sqrt(1 - alpha_bar_prev) * predicted_noise
            else:
                x = x0_pred  # final step: this is the fully generated phase

        return x

    @torch.no_grad()
    def generate_averaged(self, t00, t50, phase, volume_shape, num_samples=5,
                           guidance_scale=1.0, reduction="mean"):
        """
        Runs generate() multiple times -- each call starts from independent
        random noise, so results differ run to run -- and combines them.
        Use this instead of a single generate() call when you want a more
        stable output for TRE scoring, plus a variance estimate.

        reduction: "mean" or "median" across samples. Median is a bit more
            robust to any single bad/outlier generation.

        Returns:
            combined:  (1, 1, D, H, W) -- the reduced result
            std_map:   (1, 1, D, H, W) -- per-voxel std across samples,
                       a rough uncertainty map (high std = model unsure/
                       inconsistent in that region)
            samples:   list of the individual (1, 1, D, H, W) generations,
                       in case you want per-sample TRE and a mean +/- std
                       instead of just scoring the combined volume
        """
        samples = [
            self.generate(t00, t50, phase, volume_shape, guidance_scale=guidance_scale)
            for _ in range(num_samples)
        ]
        stacked = torch.stack(samples, dim=0)  # (num_samples, 1, 1, D, H, W)

        if reduction == "mean":
            combined = stacked.mean(dim=0)
        elif reduction == "median":
            combined = stacked.median(dim=0).values
        else:
            raise ValueError(f"Unknown reduction: {reduction!r} (use 'mean' or 'median')")

        std_map = stacked.std(dim=0)
        return combined, std_map, samples


# ---------------------------------------------------------------------------
# Self-test: confirms the sampling loop runs end-to-end on dummy data
# ---------------------------------------------------------------------------

def run_selftest():
    print("Running self-test with an untrained model and dummy anchors...\n")

    device = "cpu"
    volume_shape = (32, 32, 32)  # small on purpose, just testing plumbing

    model = UNet3DFiLM(in_channels=3, out_channels=1, base_ch=8)
    scheduler = NoiseScheduler(num_timesteps=1000, device=device)
    sampler = DDIMSampler(model, scheduler, num_steps=10, device=device)  # few steps, fast

    t00 = torch.randn((1, 1, *volume_shape))
    t50 = torch.randn((1, 1, *volume_shape))

    output = sampler.generate(t00, t50, phase=0.3, volume_shape=volume_shape)

    assert output.shape == (1, 1, *volume_shape), "Output shape mismatch!"
    assert torch.isfinite(output).all(), "Output contains NaN or Inf!"
    print(f"Output shape: {tuple(output.shape)}")
    print(f"Output value range: [{output.min().item():.3f}, {output.max().item():.3f}]")
    print("\nSelf-test passed: sampling loop runs, output shape is correct, "
          "no NaNs. (Values are meaningless here since the model is untrained "
          "-- this only confirms the mechanics work.)")


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--selftest", action="store_true")
    parser.add_argument("--checkpoint", type=str, default=None,
                         help="Path to a .pt file saved by diffusion_training.py")
    parser.add_argument("--data-root", type=str, default="dirlab_preprocessed")
    parser.add_argument("--case-id", type=int, default=1)
    parser.add_argument("--phase", type=str, default="T30", help="e.g. T30")
    parser.add_argument("--num-steps", type=int, default=50)
    parser.add_argument("--base-ch", type=int, default=16)
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--output-path", type=str, default="generated_phase.npy")
    parser.add_argument("--guidance-scale", type=float, default=1.0,
                         help="Classifier-free guidance strength. 1.0 = off. Only meaningful "
                              "if the checkpoint was trained with cond_dropout_prob > 0.")
    parser.add_argument("--num-samples", type=int, default=1,
                         help="If > 1, generates this many samples and combines them "
                              "(see --reduction). Also saves a std-map .npy alongside "
                              "the main output as a rough uncertainty estimate.")
    parser.add_argument("--reduction", type=str, default="mean", choices=["mean", "median"],
                         help="How to combine multiple samples when --num-samples > 1.")
    args = parser.parse_args()

    if args.selftest:
        run_selftest()
    else:
        if args.checkpoint is None:
            raise ValueError("--checkpoint is required outside of --selftest mode")

        t00 = np.load(os.path.join(args.data_root, f"case{args.case_id}", "T00.npy")).astype(np.float32)
        t50 = np.load(os.path.join(args.data_root, f"case{args.case_id}", "T50.npy")).astype(np.float32)

        # Data is already normalized to [-1, 1] -- pass through unchanged.
        # (Do NOT clip/1000 here -- see diffusion_training.py's norm() for why.)
        def norm(x):
            return x

        t00_t = torch.from_numpy(norm(t00)).unsqueeze(0).unsqueeze(0)
        t50_t = torch.from_numpy(norm(t50)).unsqueeze(0).unsqueeze(0)

        model = UNet3DFiLM(in_channels=3, out_channels=1, base_ch=args.base_ch)
        model.load_state_dict(torch.load(args.checkpoint, map_location=args.device))

        scheduler = NoiseScheduler(num_timesteps=1000, device=args.device)
        sampler = DDIMSampler(model, scheduler, num_steps=args.num_steps, device=args.device)

        phase_value = int(args.phase[1:]) / 100.0
        t00_dev = t00_t.to(args.device)
        t50_dev = t50_t.to(args.device)

        if args.num_samples > 1:
            combined, std_map, _ = sampler.generate_averaged(
                t00_dev, t50_dev, phase=phase_value, volume_shape=t00.shape,
                num_samples=args.num_samples, guidance_scale=args.guidance_scale,
                reduction=args.reduction,
            )
            result = combined.squeeze().cpu().numpy()  # already in [-1,1], matches saved .npy convention
            std_result = std_map.squeeze().cpu().numpy()
            np.save(args.output_path, result)
            std_path = args.output_path.replace(".npy", "_std.npy")
            np.save(std_path, std_result)
            print(f"Generated {args.phase} for case {args.case_id} "
                  f"({args.num_samples} samples, {args.reduction}-combined), "
                  f"saved to {args.output_path} (uncertainty map: {std_path})")
        else:
            generated = sampler.generate(t00_dev, t50_dev, phase=phase_value,
                                          volume_shape=t00.shape,
                                          guidance_scale=args.guidance_scale)
            result = generated.squeeze().cpu().numpy()  # already in [-1,1], matches saved .npy convention
            np.save(args.output_path, result)
            print(f"Generated {args.phase} for case {args.case_id}, saved to {args.output_path}")


# Config — paths and settings

Paths updated to match your actual Drive structure: everything lives under `CSE499/`, not
directly under `MyDrive/`.

In [ ]:
import sys
sys.path.append('/content')

import os
import time
import torch
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = '/content/drive/MyDrive/CSE499'
BASE = os.path.join(PROJECT_ROOT, 'dirlab_preprocessed')
SYNTHETIC_ROOT = os.path.join(BASE, 'synthetic')

TRAIN_CASES = [1, 2, 5, 6, 7, 8, 10]     # case4 excluded from synthetic, real-only elsewhere
SYNTHETIC_BLOCKLIST = {4}
VAL_CASES = [3, 9]

BASELINE_CHECKPOINT = os.path.join(PROJECT_ROOT, 'checkpoints_50ep', 'epoch_50.pt')  # old model, for comparison only
NEW_CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints_upgraded_scratch')       # fresh from-scratch run

# --- The 3 upgrades ---
PATCH_SIZE = 48          # was 32. Bigger patch = more context, but ~3.4x more voxels than 32^3 -> slower per epoch.
COND_DROPOUT_PROB = 0.15 # classifier-free guidance training. 0 = old behavior, no dropout.
USE_AMP = True           # mixed precision -- frees up GPU memory, needed to afford PATCH_SIZE=48+

EPOCHS = 50               # matching your original run's epoch count for a fair before/after comparison.
                           # Honest heads-up: at patch 48^3 this will likely take noticeably longer wall-clock
                           # than the original 32^3 run did. Watch your first few epochs' timing below and
                           # adjust EPOCHS down if you're tight on time before the deadline.

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Train the upgraded model from scratch

This is a fresh model — random initialization, not loaded from `checkpoints_50ep`. Same
dataset, same case split, same epoch budget as your original run, but with bigger patches,
CFG dropout, and AMP turned on. This is the version you compare against the original to show
your professor what the changes actually bought you.

The `%%time`-style print at the end tells you how long the run took, useful for judging
whether you have room to also try `PATCH_SIZE = 64` before the deadline.

In [ ]:
from unet3d_film import UNet3DFiLM
from diffusion_training import PhasePatchDataset, NoiseScheduler, train
from torch.utils.data import DataLoader

synthetic_case_ids = [c for c in TRAIN_CASES if c not in SYNTHETIC_BLOCKLIST]

train_dataset = PhasePatchDataset(
    data_root=BASE,
    case_ids=TRAIN_CASES,
    patch_size=PATCH_SIZE,
    synthetic_root=SYNTHETIC_ROOT,
    synthetic_case_ids=synthetic_case_ids,
)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
print(f"Training samples: {len(train_dataset)}")

model = UNet3DFiLM(in_channels=3, out_channels=1, base_ch=16)  # fresh random init -- NOT loaded from a checkpoint

scheduler = NoiseScheduler(num_timesteps=1000, device=device)

start = time.time()
history = train(
    model, train_loader, scheduler, device=device,
    epochs=EPOCHS, lr=1e-4,
    checkpoint_dir=NEW_CHECKPOINT_DIR,
    cond_dropout_prob=COND_DROPOUT_PROB,
    use_amp=USE_AMP,
    start_epoch=0,
)
elapsed_min = (time.time() - start) / 60
print(f"Training took {elapsed_min:.1f} minutes for {EPOCHS} epochs at patch_size={PATCH_SIZE}")

plt.plot(range(1, EPOCHS + 1), history)
plt.xlabel("Epoch")
plt.ylabel("Avg MSE loss")
plt.title(f"Loss — upgraded model, patch_size={PATCH_SIZE}, cond_dropout={COND_DROPOUT_PROB}")
plt.show()

# Load both models — baseline vs upgraded

The baseline model was trained without CFG dropout, so it must be generated with
`guidance_scale=1.0` (no guidance) — running guidance on it would push it through the
zeroed-anchor branch it never learned, which makes output worse, not better.

In [ ]:
from ddim_sampling import DDIMSampler

# Baseline (original 50-epoch, patch 32, no dropout)
baseline_model = UNet3DFiLM(in_channels=3, out_channels=1, base_ch=16)
baseline_model.load_state_dict(torch.load(BASELINE_CHECKPOINT, map_location=device))
baseline_model.to(device).eval()
baseline_sampler = DDIMSampler(baseline_model, scheduler, num_steps=50, device=device)

# Upgraded (fresh from-scratch run, patch 48, dropout 0.15)
final_epoch_path = os.path.join(NEW_CHECKPOINT_DIR, f"epoch_{EPOCHS}.pt")
upgraded_model = UNet3DFiLM(in_channels=3, out_channels=1, base_ch=16)
upgraded_model.load_state_dict(torch.load(final_epoch_path, map_location=device))
upgraded_model.to(device).eval()
upgraded_sampler = DDIMSampler(upgraded_model, scheduler, num_steps=50, device=device)

print("Both models loaded.")

# Generate the same validation phase from both — the actual comparison

Case 3, T30 (the same validation phase you've been checking all along). Baseline gets a
single plain generation. Upgraded gets guidance + 5-sample averaging — the whole point of
building those was to use them here.

In [ ]:
EVAL_CROP_SIZE = 48  # matches training patch size -- keeps both models within a resolution
                      # they've actually seen. NOTE: baseline was trained at patch 32, so it's
                      # still extrapolating a bit beyond its training resolution here -- an
                      # inherent limitation of comparing two differently-sized models, not a bug.

def center_crop(vol, size):
    d, h, w = vol.shape
    d0, h0, w0 = max(0, (d - size) // 2), max(0, (h - size) // 2), max(0, (w - size) // 2)
    crop = vol[d0:d0 + size, h0:h0 + size, w0:w0 + size]
    pad = [(0, max(0, size - crop.shape[i])) for i in range(3)]
    if any(a or b for a, b in pad):
        crop = np.pad(crop, pad, mode="constant", constant_values=0)
    return crop

case_dir = os.path.join(BASE, "case3")
t00 = center_crop(np.load(os.path.join(case_dir, "T00.npy")).astype(np.float32), EVAL_CROP_SIZE)
t50 = center_crop(np.load(os.path.join(case_dir, "T50.npy")).astype(np.float32), EVAL_CROP_SIZE)
t30_real = center_crop(np.load(os.path.join(case_dir, "T30.npy")).astype(np.float32), EVAL_CROP_SIZE)

# Data is ALREADY normalized to [-1,1] by preprocessing -- do not divide by 1000 again.
# (That was the double-normalization bug that crushed everything toward zero -- fixed now.)
def norm(x):
    return x

t00_t = torch.from_numpy(norm(t00)).unsqueeze(0).unsqueeze(0).to(device)
t50_t = torch.from_numpy(norm(t50)).unsqueeze(0).unsqueeze(0).to(device)
t30_real_norm = norm(t30_real)

print("t00 range:", t00.min(), t00.max(), "| real T30 range:", t30_real.min(), t30_real.max())
# Sanity check: these should span close to the full [-1,1] range, NOT be crushed near zero.
# If you see a range like (-0.001, 0.001) here, something upstream re-broke the normalization.

# --- Baseline: no guidance, single sample (matches how it was always used) ---
baseline_out = baseline_sampler.generate(t00_t, t50_t, phase=0.3, volume_shape=t00.shape, guidance_scale=1.0)
baseline_np = baseline_out.squeeze().cpu().numpy()
baseline_mse = np.mean((baseline_np - t30_real_norm) ** 2)

# --- Upgraded: guidance + 5-sample averaging ---
GUIDANCE_SCALE = 2.0
NUM_SAMPLES = 5
upgraded_combined, upgraded_std, _ = upgraded_sampler.generate_averaged(
    t00_t, t50_t, phase=0.3, volume_shape=t00.shape,
    num_samples=NUM_SAMPLES, guidance_scale=GUIDANCE_SCALE, reduction="mean",
)
upgraded_np = upgraded_combined.squeeze().cpu().numpy()
upgraded_std_np = upgraded_std.squeeze().cpu().numpy()
upgraded_mse = np.mean((upgraded_np - t30_real_norm) ** 2)

print(f"baseline  -> min: {baseline_np.min():.4f} max: {baseline_np.max():.4f} std: {baseline_np.std():.4f}")
print(f"upgraded  -> min: {upgraded_np.min():.4f} max: {upgraded_np.max():.4f} std: {upgraded_np.std():.4f}")
print(f"real T30  -> min: {t30_real_norm.min():.4f} max: {t30_real_norm.max():.4f} std: {t30_real_norm.std():.4f}")
print()
print(f"Baseline  (patch=32, no dropout, 1 sample):        MSE = {baseline_mse:.4f}")
print(f"Upgraded  (patch={PATCH_SIZE}, dropout={COND_DROPOUT_PROB}, guidance={GUIDANCE_SCALE}, {NUM_SAMPLES} samples): MSE = {upgraded_mse:.4f}")
print(f"Change: {((upgraded_mse - baseline_mse) / baseline_mse) * 100:+.1f}%")

# Visual comparison for your professor

Real T30 vs baseline generation vs upgraded generation, plus the upgraded model's
uncertainty map. This is the figure worth putting in a slide — numbers alone are less
convincing than seeing it side by side.

**Honest framing note:** with ~7-8 real patient anatomies underlying training either way, a
modest MSE improvement is the realistic and legitimate outcome here — that's the finding,
not a shortfall. Don't oversell this as a dramatic leap; frame it as "the architectural
changes measurably helped, within the limits of a small dataset."

In [ ]:
mid = t00.shape[0] // 2

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(t30_real_norm[mid], cmap="gray")
axes[0].set_title("Real T30 (ground truth)")

axes[1].imshow(baseline_np[mid], cmap="gray")
axes[1].set_title(f"Baseline\nMSE={baseline_mse:.4f}")

axes[2].imshow(upgraded_np[mid], cmap="gray")
axes[2].set_title(f"Upgraded (guided, averaged)\nMSE={upgraded_mse:.4f}")

im = axes[3].imshow(upgraded_std_np[mid], cmap="hot")
axes[3].set_title("Upgraded: uncertainty across samples")
plt.colorbar(im, ax=axes[3], fraction=0.046)

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "baseline_vs_upgraded_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure to {PROJECT_ROOT}/baseline_vs_upgraded_comparison.png")